In [28]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from torchvision import transforms
from torch.utils.data import DataLoader

from transformers import CLIPModel
from medmnist import INFO
from medmnist import PathMNIST

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from torch.cuda.amp import autocast, GradScaler

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce GTX 1650


In [30]:
data_flag = "pathmnist"
info = INFO[data_flag]
DataClass = getattr(__import__("medmnist"), info["python_class"])

train_dataset = DataClass(split="train", download=True)
val_dataset = DataClass(split="val", download=True)
test_dataset = DataClass(split="test", download=True)

num_classes = len(info["label"])

Using downloaded and verified file: C:\Users\Acer\.medmnist\pathmnist.npz
Using downloaded and verified file: C:\Users\Acer\.medmnist\pathmnist.npz
Using downloaded and verified file: C:\Users\Acer\.medmnist\pathmnist.npz


In [31]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.48145466,0.4578275,0.40821073],
        std=[0.26862954,0.26130258,0.27577711]
    )
])

train_dataset.transform = transform
val_dataset.transform = transform
test_dataset.transform = transform

In [32]:
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [33]:
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32",
    use_safetensors=True
)

clip_model = clip_model.to(device)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:
for param in clip_model.parameters():
    param.requires_grad = False

In [35]:
for param in clip_model.vision_model.encoder.layers[11].parameters():
    param.requires_grad = True

In [36]:
class CLIPClassifier(nn.Module):

    def __init__(self, clip_model, num_classes):
        super().__init__()

        self.clip = clip_model

        self.classifier = nn.Sequential(
            nn.Linear(768,256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256,num_classes)
        )

    def forward(self,x):

        vision_outputs = self.clip.vision_model(pixel_values=x)

        features = vision_outputs.pooler_output

        logits = self.classifier(features)

        return logits


model = CLIPClassifier(clip_model,num_classes).to(device)

In [37]:
optimizer = optim.AdamW([
    
    {"params": model.classifier.parameters(), "lr":1e-3},
    
    {"params": clip_model.vision_model.encoder.layers[11].parameters(),"lr":1e-5}
    
])

criterion = nn.CrossEntropyLoss()

scaler = GradScaler()

C:\Users\Acer\AppData\Local\Temp\ipykernel_14048\1130916294.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [38]:
def train_epoch(loader):

    model.train()

    all_preds = []
    all_labels = []

    total_loss = 0

    for images,labels in loader:

        images = images.to(device)
        labels = labels.squeeze().long().to(device)

        optimizer.zero_grad()

        with autocast():

            outputs = model(images)

            loss = criterion(outputs,labels)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        total_loss += loss.item()

        preds = outputs.argmax(1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels,all_preds)

    return total_loss/len(loader), acc

In [39]:
def evaluate(loader):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images,labels in loader:

            images = images.to(device)
            labels = labels.squeeze().long().to(device)

            outputs = model(images)

            preds = outputs.argmax(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels,all_preds)

    f1 = f1_score(all_labels,all_preds,average="macro")

    return acc,f1,all_labels,all_preds

In [40]:
import time

epochs = 5
best_val = 0

train_loss_history = []
train_acc_history = []
val_loss_history = []
val_acc_history = []

start_time = time.time()   # Start timing

for epoch in range(epochs):

    # Train
    train_loss, train_acc = train_epoch(train_loader)

    # Validate (assuming evaluate returns: loss, acc, preds, labels)
    val_loss, val_acc, _, _ = evaluate(val_loader)

    # Store history
    train_loss_history.append(train_loss)
    train_acc_history.append(train_acc)
    val_loss_history.append(val_loss)
    val_acc_history.append(val_acc)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print("-" * 40)

    # Save best model
    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), "best_clip_model.pth")

end_time = time.time()   # End timing
total_time = end_time - start_time

print(f"\nTotal training time for {epochs} epochs: {total_time:.2f} seconds")

C:\Users\Acer\AppData\Local\Temp\ipykernel_14048\1026171584.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1/5
Train Loss: 0.2955 | Train Acc: 0.8955
Val   Loss: 0.9433 | Val   Acc: 0.9430
----------------------------------------


C:\Users\Acer\AppData\Local\Temp\ipykernel_14048\1026171584.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 2/5
Train Loss: 0.1796 | Train Acc: 0.9386
Val   Loss: 0.9537 | Val   Acc: 0.9536
----------------------------------------


C:\Users\Acer\AppData\Local\Temp\ipykernel_14048\1026171584.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 3/5
Train Loss: 0.1495 | Train Acc: 0.9489
Val   Loss: 0.9506 | Val   Acc: 0.9507
----------------------------------------


C:\Users\Acer\AppData\Local\Temp\ipykernel_14048\1026171584.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 4/5
Train Loss: 0.1285 | Train Acc: 0.9562
Val   Loss: 0.9681 | Val   Acc: 0.9675
----------------------------------------


C:\Users\Acer\AppData\Local\Temp\ipykernel_14048\1026171584.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 5/5
Train Loss: 0.1139 | Train Acc: 0.9611
Val   Loss: 0.9605 | Val   Acc: 0.9597
----------------------------------------

Total training time for 5 epochs: 17029.25 seconds


In [42]:
test_acc, test_f1, y_true, y_pred = evaluate(test_loader)

print("Test Results")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")

Test Results
Test Accuracy: 0.9180
Test F1 Score: 0.8892


In [43]:
cm = confusion_matrix(y_true,y_pred)

per_class = cm.diagonal()/cm.sum(axis=1)

print("\nPer Class Accuracy")

for i,a in enumerate(per_class):

    print(f"Class {i}: {a:.4f}")


Per Class Accuracy
Class 0: 0.9664
Class 1: 1.0000
Class 2: 0.8820
Class 3: 0.9937
Class 4: 0.9671
Class 5: 0.8108
Class 6: 0.8462
Class 7: 0.5724
Class 8: 0.9513


In [45]:
import json

results = {
    "train_loss": [float(x) for x in train_loss_history],
    "train_acc": [float(x) for x in train_acc_history],
    "val_loss": [float(x) for x in val_loss_history],
    "val_acc": [float(x) for x in val_acc_history],
    "test_acc": float(test_acc),
    "test_f1": float(test_f1),
    "y_true": list(map(int, y_true)),
    "y_pred": list(map(int, y_pred))
}

with open("training_results.json", "w") as f:
    json.dump(results, f)

print("Saved successfully")

Saved successfully
